# Bukhsh pipeline

Real-life: **ssd** trim. Synthetic: **none** trim. "First (full-trace)" runs HPO inline in the cell itself, both Real and Synthetic. Half-prefix/Plain-field reuse the already-trained model.

## Real

In [ ]:
import sys
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

REAL_DATA_DIR = ROOT / "data" / "real-life"
BEST_MODELS = ROOT / "best_models"
RESULTS = ROOT / "results"
TRIM_NAME = "ssd"
SSD_TRIM_CFG = {"method": "ssd", "frac": 0.6, "k": 1.5, "pct": 0.25}

REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

print(f"ROOT = {ROOT}")
print(f"{len(REAL_DATASETS)} real-life datasets")


def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

### 1. First (full-trace)

In [ ]:
import pickle
import matplotlib.pyplot as plt
from hyperopt import tpe, Trials, hp, fmin, STATUS_OK
from sklearn.metrics import mean_absolute_error, mean_squared_error
from create_prefixes_from_windows import make_three_way_split
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from bukhsh.trainer import BukhshTrainer
from bukhsh.params  import default_params

BUKHSH_SPACE = {
    "num_heads":     [2, 4],
    "batch_size":    [16, 32],
    "epochs":        [20, 50],
    "learning_rate": [0.0005, 0.001, 0.005],
}
BUKHSH_MAX_EVAL = 12


def pt_kpi_series(event_log, test_index_cc, test_index_tt, window="days"):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr


def _bukhsh_weights_complete(work_dir, run_name):
    results = Path(work_dir) / "results"
    tasks = ("next_act", "next_role", "next_time", "rem_time")
    return all((results / f"{run_name}_{t}.weights.h5").exists() for t in tasks)


bukhsh_results = {}
for name in REAL_DATASETS:
    run_name = f"{name}_test_full"
    metrics_path = RESULTS / "bukhsh_hpo" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"

    print(f"\n{'='*60}\n{name} (bukhsh full-trace HPO)\n{'='*60}")
    if metrics_path.exists():
        print("  [skip] metrics already exist")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")

    _, _, val_as_test_df = make_three_way_split(
        df_full, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["train_split"], full_traces=True,
    )
    empty_val = pd.DataFrame(columns=train_df.columns)

    hpo_dir = BEST_MODELS / name / TRIM_NAME / "bukhsh" / "hpo_trials"
    hpo_dir.mkdir(parents=True, exist_ok=True)

    _trials_pkl = hpo_dir / "hyperopt_trials.pkl"
    if _trials_pkl.exists():
        with open(_trials_pkl, "rb") as _f:
            _hpo_trials = pickle.load(_f)
        _completed = len(_hpo_trials.trials)
        print(f"Resuming: {_completed}/{BUKHSH_MAX_EVAL} trials already done")
    else:
        _hpo_trials = Trials()
        _completed  = 0
        print(f"Bukhsh Bayesian HPO: {BUKHSH_MAX_EVAL} trials")

    bukhsh_hpo_results = []
    _trial_counter = [_completed]

    def _bukhsh_objective(trial_cfg):
        i = _trial_counter[0]
        _trial_counter[0] += 1
        trial_id  = f"trial_{i:03d}"
        trial_dir = hpo_dir / trial_id
        trial_dir.mkdir(parents=True, exist_ok=True)

        with open(_trials_pkl, "wb") as _f:
            pickle.dump(_hpo_trials, _f)

        result_path = trial_dir / "result.json"
        meta_path   = trial_dir / "meta.pkl"

        if result_path.exists():
            cached = json.loads(result_path.read_text())
            print(f"[{i+1}/{BUKHSH_MAX_EVAL}] cached  CC MAE={cached['val_cc_mae']:.4f}  ({trial_id})")
            bukhsh_hpo_results.append(cached)
            return {"loss": cached["val_cc_mae"], "status": STATUS_OK}

        print(f"\n[{i+1}/{BUKHSH_MAX_EVAL}] {trial_cfg}")
        params = default_params(**trial_cfg)
        trainer = BukhshTrainer(
            train_df, val_df=empty_val, test_df=val_as_test_df,
            run_name=trial_id, params=params, output_dir=trial_dir,
        )

        _meta_ok    = meta_path.exists()
        _weights_ok = _bukhsh_weights_complete(trial_dir, trial_id)

        if _meta_ok and _weights_ok:
            with open(meta_path, "rb") as _mf:
                _check_meta = pickle.load(_mf)
            if not all("num_heads" in _tm for _tm in _check_meta.values()):
                print("  old meta (no num_heads) — deleting partial state, will retrain")
                meta_path.unlink()
                for _wf in (trial_dir / "results").glob("*.weights.h5"):
                    _wf.unlink()
                t0 = time.perf_counter()
                trainer.run()
                train_s = round(time.perf_counter() - t0, 2)
            else:
                print("  model exists (meta + all weights OK), skipping training")
                trainer._train_csv = trial_dir / "train.csv"
                trainer._vocab_csv = trial_dir / "vocab_ref.csv"
                trainer._test_csv  = trial_dir / "test.csv"
                train_s = 0.0
        else:
            if _meta_ok and not _weights_ok:
                print("  meta exists but weights incomplete — retraining")
                meta_path.unlink()
                for _wf in (trial_dir / "results").glob("*.weights.h5"):
                    _wf.unlink()
            t0 = time.perf_counter()
            trainer.run()
            train_s = round(time.perf_counter() - t0, 2)

        event_log, _ = trainer.predict()
        event_log = event_log.copy()
        event_log["end_timestamp"] = (
            pd.to_datetime(event_log["end_timestamp"], utc=True).dt.tz_convert(None))

        cc_pred, tt_pred = pt_kpi_series(event_log, cc["val"].index, tt["val"].index)
        cc_mae = mean_absolute_error(cc["val"].to_numpy(), cc_pred)
        tt_mae = mean_absolute_error(tt["val"].to_numpy(), tt_pred)

        row = {**trial_cfg, "trial": trial_id, "val_cc_mae": round(cc_mae, 4),
               "val_tt_mae": round(tt_mae, 4), "train_s": train_s}
        bukhsh_hpo_results.append(row)
        result_path.write_text(json.dumps(row))
        print(f"  CC MAE={cc_mae:.4f}  TT MAE={tt_mae:.4f}  time={train_s:.0f}s")
        return {"loss": cc_mae, "status": STATUS_OK}

    _hyperopt_space = {k: hp.choice(k, v) for k, v in BUKHSH_SPACE.items()}
    fmin(fn=_bukhsh_objective, space=_hyperopt_space, algo=tpe.suggest,
         max_evals=BUKHSH_MAX_EVAL, trials=_hpo_trials, show_progressbar=False)

    if not bukhsh_hpo_results:
        for _rp in sorted(hpo_dir.glob("trial_*/result.json")):
            bukhsh_hpo_results.append(json.loads(_rp.read_text()))
        print(f"Loaded {len(bukhsh_hpo_results)} cached results from disk")

    bukhsh_hpo_df = pd.DataFrame(bukhsh_hpo_results).sort_values("val_cc_mae")

    best_bukhsh = bukhsh_hpo_df.iloc[0].to_dict()
    print("Best Bukhsh config:")
    for k, v in best_bukhsh.items():
        print(f"  {k}: {v}")

    best_dir = BEST_MODELS / name / TRIM_NAME / "bukhsh"
    best_dir.mkdir(parents=True, exist_ok=True)
    bukhsh_hpo_df.to_csv(best_dir / "hpo_results.csv", index=False)

    best_params = {k: best_bukhsh[k] for k in BUKHSH_SPACE}
    best_params = {k: int(v) if k in ("num_heads", "batch_size", "epochs") else
                   float(v) if k == "learning_rate" else v
                   for k, v in best_params.items()}
    (best_dir / "best_params.json").write_text(json.dumps(best_params, indent=2))
    print(f"Saved -> {best_dir}/best_params.json")

    final_bukhsh_dir = best_dir / run_name
    params_best = default_params(**best_params)
    trainer_final = BukhshTrainer(
        train_df, val_df=val_df, test_df=test_df,
        run_name=run_name, params=params_best, output_dir=final_bukhsh_dir,
    )

    _final_meta_ok    = (final_bukhsh_dir / "meta.pkl").exists()
    _final_weights_ok = _bukhsh_weights_complete(final_bukhsh_dir, run_name)

    if _final_meta_ok and _final_weights_ok:
        print(f"Final model exists (meta + all weights OK), skipping retrain -> {final_bukhsh_dir}")
        _final_train_s = 0.0
    else:
        if _final_meta_ok and not _final_weights_ok:
            print("  Final model: meta exists but weights incomplete — retraining")
        t0 = time.perf_counter()
        trainer_final.run()
        _final_train_s = round(time.perf_counter() - t0, 2)
        print(f"Final Bukhsh retrain: {_final_train_s:.1f}s")

    print(f"Saved -> {final_bukhsh_dir}")

    pd.DataFrame([
        dict(model="bukhsh_hpo", phase="train", params_json=json.dumps(best_params),
             val_mse=float("nan"), time_s=_final_train_s, is_best=True,
             dataset=run_name, series="all"),
    ]).to_csv(best_dir / f"time_{run_name}.csv", index=False)
    print(f"Timing saved -> {best_dir}/time_{run_name}.csv")

    cc_actual = cc["test"].to_numpy()
    tt_actual = tt["test"].to_numpy()

    t0 = time.perf_counter()
    event_log_b, rem_time_df_b = trainer_final.predict()
    predict_s_b = round(time.perf_counter() - t0, 2)

    event_log_b = event_log_b.copy()
    event_log_b["end_timestamp"] = (
        pd.to_datetime(event_log_b["end_timestamp"], utc=True).dt.tz_convert(None))

    cc_pred_b, tt_pred_b = pt_kpi_series(
        event_log_b, test_index_cc=cc["test"].index, test_index_tt=tt["test"].index)

    out_dir_b = metrics_path.parent
    out_dir_b.mkdir(parents=True, exist_ok=True)

    results_b = {
        "concurrent_cases": {"mse": mean_squared_error(cc_actual, cc_pred_b),
                             "mae": mean_absolute_error(cc_actual, cc_pred_b)},
        "throughput_time":  {"mse": mean_squared_error(tt_actual, tt_pred_b),
                             "mae": mean_absolute_error(tt_actual, tt_pred_b)},
    }
    bukhsh_results[name] = results_b
    pd.DataFrame([
        dict(dataset=run_name, series=s, model="bukhsh_hpo", mse=m["mse"], mae=m["mae"])
        for s, m in results_b.items()
    ]).to_csv(out_dir_b / f"metrics_{run_name}.csv", index=False)

    _has_ts = "anchor_timestamp" in rem_time_df_b.columns
    if _has_ts:
        rt_df = rem_time_df_b.copy()
        rt_df["start_timestamp"]  = pd.to_datetime(rt_df["start_timestamp"])
        rt_df["anchor_timestamp"] = pd.to_datetime(rt_df["anchor_timestamp"])
    else:
        _ts   = pd.to_datetime(test_df["end_timestamp"], utc=True).dt.tz_convert(None)
        _tdf  = test_df.assign(_ts=_ts).sort_values(["caseid", "_ts"])
        _start  = _tdf.groupby("caseid")["_ts"].first().rename("start_timestamp").reset_index()
        _anchor = _tdf.groupby("caseid")["_ts"].last().rename("anchor_timestamp").reset_index()
        rt_df = rem_time_df_b.merge(_start, on="caseid").merge(_anchor, on="caseid")
    rt_df["predicted_end"] = (rt_df["anchor_timestamp"]
                              + pd.to_timedelta(rt_df["rem_time_days"], unit="D"))
    rt_log_b = pd.concat([
        rt_df[["caseid", "start_timestamp"]].rename(columns={"start_timestamp": "end_timestamp"}),
        rt_df[["caseid", "predicted_end"]].rename(columns={"predicted_end": "end_timestamp"}),
    ], ignore_index=True)
    cc_pred_b_rt, tt_pred_b_rt = pt_kpi_series(
        rt_log_b, test_index_cc=cc["test"].index, test_index_tt=tt["test"].index)
    results_b_rt = {
        "concurrent_cases": {"mse": mean_squared_error(cc_actual, cc_pred_b_rt),
                             "mae": mean_absolute_error(cc_actual, cc_pred_b_rt)},
        "throughput_time":  {"mse": mean_squared_error(tt_actual, tt_pred_b_rt),
                             "mae": mean_absolute_error(tt_actual, tt_pred_b_rt)},
    }
    pd.DataFrame([
        dict(dataset=run_name, series=s, model="bukhsh_hpo_rt", mse=m["mse"], mae=m["mae"])
        for s, m in results_b_rt.items()
    ]).to_csv(out_dir_b / f"metrics_{run_name}_rt.csv", index=False)

    for series_name, actual, pred, index in [
        ("concurrent_cases", cc_actual, cc_pred_b, cc["test"].index),
        ("throughput_time",  tt_actual, tt_pred_b, tt["test"].index),
    ]:
        m = results_b[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color="green",  label="actual",            linewidth=1.5)
        ax.plot(index, pred,   color="purple", label="Bukhsh-HPO (pred)", linestyle="--")
        ax.set_title(f"{run_name} — {series_name}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}")
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_b / f"{series_name}.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close()
    print(f"  [Bukhsh] {predict_s_b:.1f}s  ->  {out_dir_b}")


### 2. Half-prefix

In [ ]:
from run_predictions_real import _run_bukhsh

for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (bukhsh half-prefix)\n{'='*60}")
    run_name = f"{name}_test_full"
    bmd = BEST_MODELS / name / TRIM_NAME / "bukhsh"
    best_params_path = bmd / "best_params.json"
    if not best_params_path.exists():
        print(f"  [skip] no trained model at {best_params_path}")
        continue

    half_metrics = RESULTS / "bukhsh_hpo_half" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"
    if half_metrics.exists():
        print("  [skip] half-prefix metrics already exist")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")
    best_params = json.loads(best_params_path.read_text())
    _run_bukhsh("half", bmd, run_name, TRIM_NAME, best_params,
                train_df, val_df, test_df, df_full, cc, tt)

### 3. Plain-field

In [ ]:
from runner import (
    get_inflight_cases, build_sos_cases, predict_bukhsh_plain_field,
    compute_cc_tt_metrics, save_pf_metrics, save_pf_plot, _rem_time_to_event_log,
)
from sos import most_frequent_first_activity, most_frequent_first_resource, empirical_arrival_hour_sampler
from arrival import compute_arrival_series, ProphetArrivalModel
from bukhsh.params import default_params as bukhsh_dp
from bukhsh.trainer import BukhshTrainer

for name in REAL_DATASETS:
    run_name = f"{name}_test_full"
    print(f"\n{'='*60}\n{name} (bukhsh plain-field)\n{'='*60}")

    bdir = BEST_MODELS / name / TRIM_NAME / "bukhsh" / run_name
    pf_dir = bdir / "pf"
    raw_done = (pf_dir / "rem_time.csv").exists() and (pf_dir / "event_log.csv").exists()
    metrics_done = ((RESULTS / "plain_field" / "bukhsh_suffix" / TRIM_NAME / run_name / f"metrics_{run_name}.csv").exists()
                    and (RESULTS / "plain_field" / "bukhsh_rt" / TRIM_NAME / run_name / f"metrics_{run_name}.csv").exists())
    if raw_done and metrics_done:
        print("  [skip] raw predictions + metrics already saved")
        continue

    if not (bdir / "meta.pkl").exists():
        print(f"  [skip] no trained model at {bdir}")
        continue

    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")
    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")


    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals[arrivals.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )

    trainer_b = BukhshTrainer(train_df, val_df, test_df, name, bukhsh_dp(epochs=50), output_dir=bdir)
    trainer_b._load_all_models()
    rem_time_df, suffix_log = predict_bukhsh_plain_field(trainer_b, sos_df, inflight_df)

    pf_dir.mkdir(parents=True, exist_ok=True)
    rem_time_df.to_csv(pf_dir / "rem_time.csv", index=False)
    suffix_log.to_csv(pf_dir / "event_log.csv", index=False)

    rt_log = _rem_time_to_event_log(rem_time_df)
    cc_p_rt, tt_p_rt, m_rt = compute_cc_tt_metrics(rt_log, cc["test"], tt["test"])
    out_rt = RESULTS / "plain_field" / "bukhsh_rt" / TRIM_NAME / run_name
    save_pf_metrics(name, "bukhsh_rt", m_rt, out_rt)
    save_pf_plot(cc_p_rt, tt_p_rt, cc["test"], tt["test"], out_rt, name, TRIM_NAME, "bukhsh_rt")

    cc_p, tt_p, m = compute_cc_tt_metrics(suffix_log, cc["test"], tt["test"])
    out = RESULTS / "plain_field" / "bukhsh_suffix" / TRIM_NAME / run_name
    save_pf_metrics(name, "bukhsh_suffix", m, out)
    save_pf_plot(cc_p, tt_p, cc["test"], tt["test"], out, name, TRIM_NAME, "bukhsh_suffix")
    print(f"  [bukhsh pf] cc_mae={m['cc_mae']:.2f} (suffix) / {m_rt['cc_mae']:.2f} (rt)")


## Synthetic

In [ ]:
import warnings, math
warnings.filterwarnings('ignore')

import pickle
import matplotlib.pyplot as plt
from hyperopt import tpe, Trials, hp, fmin, STATUS_OK
from sklearn.metrics import mean_absolute_error, mean_squared_error
from time_series_preprocessing import ts_splits_from_log
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from create_prefixes_from_windows import load_event_log, make_three_way_split
from setttings import set_global_seed
from bukhsh.trainer import BukhshTrainer
from bukhsh.params  import default_params

_COLS = {
    'case:concept:name':    'caseid',
    'concept:name':         'task',
    'lifecycle:transition': 'event_type',
    'org:resource':         'user',
    'time:timestamp':       'end_timestamp',
}

def pt_kpi_series(event_log, test_index_cc, test_index_tt, window='days'):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

def _bukhsh_weights_complete(work_dir, run_name):
    """True iff all 4 weight files exist for a trained Bukhsh model."""
    results = Path(work_dir) / 'results'
    tasks = ('next_act', 'next_role', 'next_time', 'rem_time')
    return all((results / f'{run_name}_{t}.weights.h5').exists() for t in tasks)

set_global_seed(1904)

_EXCLUDE_DATASETS = {'loan_recency', 'o2c_recency'}
SYNTH_DATASETS = sorted(
    p for p in (ROOT / 'data' / 'synthetic').glob('*.xes')
    if p.stem not in _EXCLUDE_DATASETS
)
print(f'{len(SYNTH_DATASETS)} synthetic datasets: {[p.stem for p in SYNTH_DATASETS]}')


### 1. First (full-trace)

In [ ]:
BUKHSH_SPACE = {
    'num_heads':     [2, 4],
    'batch_size':    [16, 32],
    'epochs':        [20, 50],
    'learning_rate': [0.0005, 0.001, 0.005],
}
BUKHSH_MAX_EVAL = 12

for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME = f'{DATASET_NAME}_test_full'
    print(f"\n{'='*60}\n{DATASET_NAME} (bukhsh synthetic HPO + first)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5,
        trim_frac=0.60, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    train_split = cc['train_split']
    val_split   = cc['val_split']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk') if 'user' in df.columns else 'unk'

    train_df, val_df, test_df = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )


    _, _, val_as_test_df = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=train_split, full_traces=True,
    )
    empty_val = pd.DataFrame(columns=train_df.columns)

    hpo_dir = BEST_MODELS / DATASET_NAME / 'bukhsh' / 'hpo_trials'
    hpo_dir.mkdir(parents=True, exist_ok=True)

    _trials_pkl = hpo_dir / "hyperopt_trials.pkl"
    if _trials_pkl.exists():
        with open(_trials_pkl, "rb") as _f:
            _hpo_trials = pickle.load(_f)
        _completed = len(_hpo_trials.trials)
        print(f'Resuming: {_completed}/{BUKHSH_MAX_EVAL} trials already done')
    else:
        _hpo_trials = Trials()
        _completed  = 0
        print(f'Bukhsh Bayesian HPO: {BUKHSH_MAX_EVAL} trials')

    bukhsh_hpo_results = []
    _trial_counter = [_completed]

    def _bukhsh_objective(trial_cfg):
        i = _trial_counter[0]
        _trial_counter[0] += 1
        trial_id  = f'trial_{i:03d}'
        trial_dir = hpo_dir / trial_id
        trial_dir.mkdir(parents=True, exist_ok=True)

        with open(_trials_pkl, "wb") as _f:
            pickle.dump(_hpo_trials, _f)

        result_path = trial_dir / "result.json"
        meta_path   = trial_dir / "meta.pkl"

        if result_path.exists():
            cached = json.loads(result_path.read_text())
            print(f'[{i+1}/{BUKHSH_MAX_EVAL}] cached  CC MAE={cached["val_cc_mae"]:.4f}  ({trial_id})')
            bukhsh_hpo_results.append(cached)
            return {'loss': cached['val_cc_mae'], 'status': STATUS_OK}

        print(f'\n[{i+1}/{BUKHSH_MAX_EVAL}] {trial_cfg}')
        params = default_params(**trial_cfg)
        trainer = BukhshTrainer(
            train_df, val_df=empty_val, test_df=val_as_test_df,
            run_name=trial_id, params=params, output_dir=trial_dir,
        )

        _meta_ok    = meta_path.exists()
        _weights_ok = _bukhsh_weights_complete(trial_dir, trial_id)

        if _meta_ok and _weights_ok:
            with open(meta_path, 'rb') as _mf:
                _check_meta = pickle.load(_mf)
            if not all('num_heads' in _tm for _tm in _check_meta.values()):
                print('  old meta (no num_heads) — deleting partial state, will retrain')
                meta_path.unlink()
                for _wf in (trial_dir / 'results').glob('*.weights.h5'):
                    _wf.unlink()
                t0 = time.perf_counter()
                trainer.run()
                train_s = round(time.perf_counter() - t0, 2)
            else:
                print('  model exists (meta + all weights OK), skipping training')
                trainer._train_csv = trial_dir / 'train.csv'
                trainer._vocab_csv = trial_dir / 'vocab_ref.csv'
                trainer._test_csv  = trial_dir / 'test.csv'
                train_s = 0.0
        else:
            if _meta_ok and not _weights_ok:
                print('  meta exists but weights incomplete — retraining')
                meta_path.unlink()
                for _wf in (trial_dir / 'results').glob('*.weights.h5'):
                    _wf.unlink()
            t0 = time.perf_counter()
            trainer.run()
            train_s = round(time.perf_counter() - t0, 2)

        event_log, _ = trainer.predict()
        event_log = event_log.copy()
        event_log['end_timestamp'] = (
            pd.to_datetime(event_log['end_timestamp'], utc=True).dt.tz_convert(None))

        cc_pred, tt_pred = pt_kpi_series(event_log, cc['val'].index, tt['val'].index)
        cc_mae = mean_absolute_error(cc['val'].to_numpy(), cc_pred)
        tt_mae = mean_absolute_error(tt['val'].to_numpy(), tt_pred)

        row = {**trial_cfg, 'trial': trial_id, 'val_cc_mae': round(cc_mae, 4),
               'val_tt_mae': round(tt_mae, 4), 'train_s': train_s}
        bukhsh_hpo_results.append(row)
        result_path.write_text(json.dumps(row))
        print(f'  CC MAE={cc_mae:.4f}  TT MAE={tt_mae:.4f}  time={train_s:.0f}s')
        return {'loss': cc_mae, 'status': STATUS_OK}

    _hyperopt_space = {k: hp.choice(k, v) for k, v in BUKHSH_SPACE.items()}
    fmin(fn=_bukhsh_objective, space=_hyperopt_space, algo=tpe.suggest,
         max_evals=BUKHSH_MAX_EVAL, trials=_hpo_trials, show_progressbar=False)


    if not bukhsh_hpo_results:
        for _rp in sorted(hpo_dir.glob("trial_*/result.json")):
            bukhsh_hpo_results.append(json.loads(_rp.read_text()))
        print(f'Loaded {len(bukhsh_hpo_results)} cached results from disk')

    bukhsh_hpo_df = pd.DataFrame(bukhsh_hpo_results).sort_values('val_cc_mae')

    best_bukhsh = bukhsh_hpo_df.iloc[0].to_dict()
    print('Best Bukhsh config:')
    for k, v in best_bukhsh.items():
        print(f'  {k}: {v}')

    best_dir = BEST_MODELS / DATASET_NAME / 'bukhsh'
    best_dir.mkdir(parents=True, exist_ok=True)
    bukhsh_hpo_df.to_csv(best_dir / 'hpo_results.csv', index=False)

    best_params = {k: best_bukhsh[k] for k in BUKHSH_SPACE}
    best_params = {k: int(v) if k in ('num_heads', 'batch_size', 'epochs') else
                   float(v) if k == 'learning_rate' else v
                   for k, v in best_params.items()}
    (best_dir / 'best_params.json').write_text(json.dumps(best_params, indent=2))
    print(f'Saved -> {best_dir}/best_params.json')

    final_bukhsh_dir = best_dir / RUN_NAME
    params_best = default_params(**best_params)
    trainer_final = BukhshTrainer(
        train_df, val_df=val_df, test_df=test_df,
        run_name=RUN_NAME, params=params_best, output_dir=final_bukhsh_dir,
    )

    _final_meta_ok    = (final_bukhsh_dir / 'meta.pkl').exists()
    _final_weights_ok = _bukhsh_weights_complete(final_bukhsh_dir, RUN_NAME)

    if _final_meta_ok and _final_weights_ok:
        print(f'Final model exists (meta + all weights OK), skipping retrain -> {final_bukhsh_dir}')
        _final_train_s = 0.0
    else:
        if _final_meta_ok and not _final_weights_ok:
            print('  Final model: meta exists but weights incomplete — retraining')
        t0 = time.perf_counter()
        trainer_final.run()
        _final_train_s = round(time.perf_counter() - t0, 2)
        print(f'Final Bukhsh retrain: {_final_train_s:.1f}s')

    print(f'Saved -> {final_bukhsh_dir}')

    pd.DataFrame([
        dict(model='bukhsh_hpo', phase='train', params_json=json.dumps(best_params),
             val_mse=float('nan'), time_s=_final_train_s, is_best=True,
             dataset=RUN_NAME, series='all'),
    ]).to_csv(best_dir / f'time_{RUN_NAME}.csv', index=False)
    print(f'Timing saved -> {best_dir}/time_{RUN_NAME}.csv')

    metrics_path = RESULTS / 'bukhsh_hpo' / 'none' / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    if metrics_path.exists():
        print(f'  metrics already exist -> {metrics_path}')
        continue

    cc_actual = cc['test'].to_numpy()
    tt_actual = tt['test'].to_numpy()

    t0 = time.perf_counter()
    event_log_b, rem_time_df_b = trainer_final.predict()
    predict_s_b = round(time.perf_counter() - t0, 2)

    event_log_b = event_log_b.copy()
    event_log_b['end_timestamp'] = (
        pd.to_datetime(event_log_b['end_timestamp'], utc=True).dt.tz_convert(None))

    cc_pred_b, tt_pred_b = pt_kpi_series(
        event_log_b, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)

    out_dir_b = metrics_path.parent
    out_dir_b.mkdir(parents=True, exist_ok=True)

    results_b = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_b),
                             'mae': mean_absolute_error(cc_actual, cc_pred_b)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_b),
                             'mae': mean_absolute_error(tt_actual, tt_pred_b)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='bukhsh_hpo', mse=m['mse'], mae=m['mae'])
        for s, m in results_b.items()
    ]).to_csv(out_dir_b / f'metrics_{RUN_NAME}.csv', index=False)

    _has_ts = 'anchor_timestamp' in rem_time_df_b.columns
    if _has_ts:
        rt_df = rem_time_df_b.copy()
        rt_df['start_timestamp']  = pd.to_datetime(rt_df['start_timestamp'])
        rt_df['anchor_timestamp'] = pd.to_datetime(rt_df['anchor_timestamp'])
    else:
        _ts   = pd.to_datetime(test_df['end_timestamp'], utc=True).dt.tz_convert(None)
        _tdf  = test_df.assign(_ts=_ts).sort_values(['caseid', '_ts'])
        _start  = _tdf.groupby('caseid')['_ts'].first().rename('start_timestamp').reset_index()
        _anchor = _tdf.groupby('caseid')['_ts'].last().rename('anchor_timestamp').reset_index()
        rt_df = rem_time_df_b.merge(_start, on='caseid').merge(_anchor, on='caseid')
    rt_df['predicted_end'] = (rt_df['anchor_timestamp']
                              + pd.to_timedelta(rt_df['rem_time_days'], unit='D'))
    rt_log_b = pd.concat([
        rt_df[['caseid', 'start_timestamp']].rename(columns={'start_timestamp': 'end_timestamp'}),
        rt_df[['caseid', 'predicted_end']].rename(columns={'predicted_end': 'end_timestamp'}),
    ], ignore_index=True)
    cc_pred_b_rt, tt_pred_b_rt = pt_kpi_series(
        rt_log_b, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)
    results_b_rt = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_b_rt),
                             'mae': mean_absolute_error(cc_actual, cc_pred_b_rt)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_b_rt),
                             'mae': mean_absolute_error(tt_actual, tt_pred_b_rt)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='bukhsh_hpo_rt', mse=m['mse'], mae=m['mae'])
        for s, m in results_b_rt.items()
    ]).to_csv(out_dir_b / f'metrics_{RUN_NAME}_rt.csv', index=False)

    for series_name, actual, pred, index in [
        ('concurrent_cases', cc_actual, cc_pred_b, cc['test'].index),
        ('throughput_time',  tt_actual, tt_pred_b, tt['test'].index),
    ]:
        m = results_b[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',            linewidth=1.5)
        ax.plot(index, pred,   color='purple', label='Bukhsh-HPO (pred)', linestyle='--')
        ax.set_title(f'{RUN_NAME} — {series_name}  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_b / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    print(f'  [Bukhsh] {predict_s_b:.1f}s  ->  {out_dir_b}')


### 2. Half-prefix

In [ ]:
for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME = f'{DATASET_NAME}_test_full'

    metrics_path = RESULTS / 'bukhsh_hpo_half' / 'none' / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    if metrics_path.exists():
        print(f'[skip] {DATASET_NAME}: half-prefix metrics already exist')
        continue

    best_params_path = BEST_MODELS / DATASET_NAME / 'bukhsh' / 'best_params.json'
    if not best_params_path.exists():
        print(f'[skip] {DATASET_NAME}: no trained model')
        continue

    print(f"\n{'='*60}\n{DATASET_NAME} (bukhsh synthetic half-prefix)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5,
        trim_frac=0.60, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk')

    train_split = cc['train_split']
    val_split   = cc['val_split']
    train_df, val_df, test_df_std = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )

    _test_case_ids = set(test_df_std['caseid'].astype(str))
    _df_test_full  = (
        df[df['caseid'].astype(str).isin(_test_case_ids)]
        .copy()
        .sort_values(['caseid', 'end_timestamp'])
    )
    _half_parts = []
    for _cid, _grp in _df_test_full.groupby('caseid', sort=False):
        _n = len(_grp)
        _half_parts.append(_grp.iloc[: max(1, math.ceil(_n / 2))])
    test_df = pd.concat(_half_parts, ignore_index=True)

    cc_actual = cc['test'].to_numpy()
    tt_actual = tt['test'].to_numpy()

    best_params      = json.loads(best_params_path.read_text())
    final_bukhsh_dir = BEST_MODELS / DATASET_NAME / 'bukhsh' / RUN_NAME
    half_bukhsh_dir  = final_bukhsh_dir / 'half_prefix'
    half_bukhsh_dir.mkdir(parents=True, exist_ok=True)

    _el_path = half_bukhsh_dir / 'event_log.csv'
    _rt_path = half_bukhsh_dir / 'rem_time.csv'

    params_b  = default_params(**best_params)
    trainer_b = BukhshTrainer(
        train_df, val_df=val_df, test_df=test_df,
        run_name=RUN_NAME, params=params_b, output_dir=final_bukhsh_dir,
    )

    if _el_path.exists() and _rt_path.exists():
        event_log_b   = pd.read_csv(_el_path)
        rem_time_df_b = pd.read_csv(_rt_path)
        predict_s_b   = 0.0
        print('  loaded cached predictions')
    else:
        t0 = time.perf_counter()
        event_log_b, rem_time_df_b = trainer_b.predict()
        predict_s_b = round(time.perf_counter() - t0, 2)
        event_log_b.to_csv(_el_path,   index=False)
        rem_time_df_b.to_csv(_rt_path, index=False)

    event_log_b = event_log_b.copy()
    event_log_b['end_timestamp'] = (
        pd.to_datetime(event_log_b['end_timestamp'], utc=True).dt.tz_convert(None))

    cc_pred_b, tt_pred_b = pt_kpi_series(
        event_log_b, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)

    out_dir_b = metrics_path.parent
    out_dir_b.mkdir(parents=True, exist_ok=True)

    results_b = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_b),
                             'mae': mean_absolute_error(cc_actual, cc_pred_b)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_b),
                             'mae': mean_absolute_error(tt_actual, tt_pred_b)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='bukhsh_hpo_half', mse=m['mse'], mae=m['mae'])
        for s, m in results_b.items()
    ]).to_csv(out_dir_b / f'metrics_{RUN_NAME}.csv', index=False)
    pd.DataFrame([
        dict(model='bukhsh_hpo_half', phase='predict', params_json=json.dumps(best_params),
             val_mse=float('nan'), time_s=predict_s_b, is_best=True,
             dataset=RUN_NAME, series='all'),
    ]).to_csv(out_dir_b / f'time_{RUN_NAME}.csv', index=False)

    _has_ts = 'anchor_timestamp' in rem_time_df_b.columns
    if _has_ts:
        rt_df = rem_time_df_b.copy()
        rt_df['start_timestamp']  = pd.to_datetime(rt_df['start_timestamp'])
        rt_df['anchor_timestamp'] = pd.to_datetime(rt_df['anchor_timestamp'])
    else:
        _ts   = pd.to_datetime(test_df['end_timestamp'], utc=True).dt.tz_convert(None)
        _tdf  = test_df.assign(_ts=_ts).sort_values(['caseid', '_ts'])
        _start  = _tdf.groupby('caseid')['_ts'].first().rename('start_timestamp').reset_index()
        _anchor = _tdf.groupby('caseid')['_ts'].last().rename('anchor_timestamp').reset_index()
        rt_df = rem_time_df_b.merge(_start, on='caseid').merge(_anchor, on='caseid')
    rt_df['predicted_end'] = (rt_df['anchor_timestamp']
                              + pd.to_timedelta(rt_df['rem_time_days'], unit='D'))
    rt_log_b = pd.concat([
        rt_df[['caseid', 'start_timestamp']].rename(columns={'start_timestamp': 'end_timestamp'}),
        rt_df[['caseid', 'predicted_end']].rename(columns={'predicted_end': 'end_timestamp'}),
    ], ignore_index=True)
    cc_pred_b_rt, tt_pred_b_rt = pt_kpi_series(
        rt_log_b, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)
    results_b_rt = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_b_rt),
                             'mae': mean_absolute_error(cc_actual, cc_pred_b_rt)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_b_rt),
                             'mae': mean_absolute_error(tt_actual, tt_pred_b_rt)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='bukhsh_hpo_half_rt', mse=m['mse'], mae=m['mae'])
        for s, m in results_b_rt.items()
    ]).to_csv(out_dir_b / f'metrics_{RUN_NAME}_rt.csv', index=False)

    for series_name, actual, pred, index in [
        ('concurrent_cases', cc_actual, cc_pred_b, cc['test'].index),
        ('throughput_time',  tt_actual, tt_pred_b, tt['test'].index),
    ]:
        m = results_b[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',                        linewidth=1.5)
        ax.plot(index, pred,   color='purple', label='Bukhsh-HPO half-prefix (pred)', linestyle='--')
        ax.set_title(f'{RUN_NAME} — {series_name}  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}  [half-prefix]')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_b / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    print(f'  {predict_s_b:.1f}s  ->  {out_dir_b}')


### 3. Plain-field

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "plain-field"))

from pf_lukas_prediction import run_pf_job, SYNTH_LOGS

for ds in SYNTH_LOGS:
    run_pf_job(ds, "none", is_real=False, do_amiri=False, do_camargo=False)
